# Clase 5 — Práctica Guiada
## ¿Cuánto vendimos, de verdad?
### Maestría en Fintech · ITBA · 2026

---

**Esta notebook la resolvemos juntos en clase.** Después vas a tener una segunda notebook
—la de Práctica Individual— con preguntas nuevas sobre los mismos datos, para resolver solo.

**La pregunta de hoy:**
> *El directorio pide el TPV 2024 (cuánta plata pasó por nuestras tarjetas en el año) para el cierre.
> Finanzas te pasa el export del procesador. ¿Qué número mandás?*

En las clases 3 y 4 trabajamos con una tabla de transacciones **prolija**: todas las claves coincidían y no faltaba nada.
Hoy trabajamos con **las mismas transacciones, tal como llegan en la vida real**.

---
## Antes de empezar: guardá tu copia

Abriste esta notebook desde el link del curso, así que **nadie pisa el trabajo de nadie**.
Pero lo que escribas acá **no se guarda**: si cerrás la pestaña, se pierde.

Para quedarte con tu trabajo: **`Archivo`** → **`Guardar una copia en Drive`**.
Te sugerimos el nombre **`Clase5_Guiada_TuNombre`**.

---
## Setup

Ejecutá la celda con **`Shift + Enter`**.

Además de las librerías de siempre, armamos una **bitácora**: una lista donde vamos a anotar cada decisión de limpieza
y cuánto movió el número. Al final de la notebook la miramos entera.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

# La bitácora de limpieza: cada paso anota qué decidimos y cómo quedó el TPV
bitacora = []

def anotar(paso, decision, datos):
    bitacora.append({
        'paso': paso,
        'decisión': decision,
        'filas': len(datos),
        'TPV ($M)': round(datos['monto_ars'].sum() / 1e6, 1),
    })

print('Librerías cargadas correctamente')

---
# Paso 1 · El número del lunes

Cargamos el archivo y hacemos lo que haría cualquiera un lunes a la mañana con el directorio esperando: **sumar**.

In [ ]:
# El archivo se descarga solo desde el repo del curso.
DATOS = 'https://raw.githubusercontent.com/camilojaure/itba-pad/main/datasets/'

sucio = pd.read_csv(DATOS + 'transacciones_sucias.csv')

print(f'{len(sucio):,} filas · {sucio.shape[1]} columnas')
sucio.head()

In [ ]:
tpv_lunes = sucio['monto_ars'].sum()
print(f'TPV 2024 según el archivo: ${tpv_lunes / 1e6:,.0f} M')

anotar('0 · Archivo original', 'Nada: el número del lunes', sucio)

**Un dato para tener en la cabeza:** en las clases 3 y 4, la tabla de transacciones **conciliada por Contabilidad**
(mismo negocio, mismo año, mismos clientes) daba un TPV de **$1.774 M**.

Este archivo dice **$2.018 M**. Son **244 millones de diferencia**, un 14%. ¿Quién tiene razón?

Antes de contestar, pidamos el segundo número que quiere el directorio: el **ticket promedio por categoría**.

In [ ]:
ticket = sucio.groupby('categoria')['monto_ars'].mean().sort_values(ascending=False)

print(f'El reporte tiene {len(ticket)} categorías')
ticket.head(15)

**Cincuenta categorías**, cuando el negocio tiene **diez**. `Supermercado`, `SUPERMERCADO` y `  Supermercado` son,
para pandas, tres categorías distintas.

> Este reporte no se puede mandar. Y fijate lo importante: **el código no tiró ningún error.**
> Como el join que multiplicaba filas en la Clase 4, el problema no avisa. Hay que ir a buscarlo.

---
# Paso 2 · El checklist: diagnóstico antes que limpieza

Antes de tocar nada, **mirar**. Son siempre las mismas preguntas, en el mismo orden:

| Pregunta | Código |
|---|---|
| ¿Qué tamaño tiene? ¿Cada columna es del tipo que debería? | `df.shape`, `df.dtypes` |
| ¿Dónde faltan datos? | `df.isna().sum()` |
| ¿Hay filas repetidas? | `df.duplicated().sum()` |
| ¿Los textos dicen lo que deberían? | `df['col'].value_counts()` |
| ¿Los números son plausibles? | `df.describe()` |

In [ ]:
print(sucio.shape)
sucio.dtypes

In [ ]:
nulos = pd.DataFrame({
    'nulos': sucio.isna().sum(),
    '%': (sucio.isna().mean() * 100).round(1),
})
nulos

In [ ]:
print(f'Filas duplicadas (idénticas en todas las columnas): {sucio.duplicated().sum():,}')

In [ ]:
sucio['tipo'].value_counts()

In [ ]:
sucio['monto_ars'].describe().round(0)

**El diagnóstico, en una lista:**

1. **`fecha` es texto**, no fecha (lo mismo que en la Clase 3)
2. **Nulos** en tres columnas: 7% en `monto_ars`, 5% en `categoria`, 4% en `fecha`
3. **869 filas duplicadas**
4. **Texto inconsistente:** 11 formas de escribir dos tipos de tarjeta (`débito`, `DÉBITO`, `credit`, ` crédito`…)
5. **Montos imposibles:** hay **negativos**, y el máximo es **$57,5 millones** cuando la mediana es $35.000

> Seis líneas de código, y ya sabemos más del archivo que quien lo mandó.
> **Diagnóstico primero, limpieza después.** Nunca al revés.

---
# Paso 3 · Duplicados: contar bien antes de borrar

Antes de borrar, la pregunta de negocio: **¿es un error de carga, o al cliente le cobraron dos veces?**
En un banco, lo segundo no se borra: se investiga.

In [ ]:
print(f"Filas idénticas:                {sucio.duplicated().sum():,}")
print(f"Filas con un transaccion_id ya visto: {sucio.duplicated(subset='transaccion_id').sum():,}")

# keep=False marca TODAS las filas del grupo, no solo la repetición: sirve para mirarlas juntas
sucio[sucio.duplicated(keep=False)].sort_values('transaccion_id').head(6)

**Cómo se lee:** las 869 repeticiones tienen **el mismo `transaccion_id`**. El procesador le pone un ID único a cada operación,
así que no son dos compras: es **la misma compra cargada dos veces**. Error de carga: se eliminan.

> Si hubiéramos encontrado **dos IDs distintos** con el mismo cliente, la misma fecha y el mismo monto, la historia sería otra:
> eso puede ser un **doble cobro**, y va a reclamos, no a la papelera.

In [ ]:
df = sucio.drop_duplicates().copy()

print(f'{len(sucio):,} → {len(df):,} filas')
anotar('1 · Duplicados', 'Eliminados: mismo transaccion_id = carga duplicada', df)

---
# Paso 4 · Texto: el enemigo invisible

Dos operaciones, **en este orden**: sacar los espacios de los costados (`strip`) y unificar mayúsculas (`capitalize`: primera letra mayúscula, el resto minúscula).

¿Por qué importa el orden? Mirá qué pasa si te salteás el `strip`:

In [ ]:
ejemplo = pd.Series(['Supermercado', 'SUPERMERCADO', '  supermercado ', 'supermercado'])

print('Sin strip: ', ejemplo.str.capitalize().unique().tolist())
print('Con strip: ', ejemplo.str.strip().str.capitalize().unique().tolist())

Sin `strip`, **el espacio de adelante sobrevive** y `capitalize` ni se entera (la primera "letra" es un espacio).
Ahora sí, sobre la tabla:

In [ ]:
df['categoria'] = df['categoria'].str.strip().str.capitalize()

df['tipo'] = (df['tipo'].str.strip().str.lower()
              .replace({'credito': 'crédito', 'credit': 'crédito'}))

print(df['tipo'].value_counts())
print()
print(df['categoria'].value_counts(dropna=False))

**De 50 categorías a 10** (más los nulos, que tratamos en el Paso 6). De 11 tipos a 2.

> `.str.title()` también sirve (pone mayúscula a **cada palabra**) y necesita el `strip` antes, igual que `capitalize`.
> Acá usamos `capitalize` porque `title` convierte `e-commerce` en `E-Commerce`, y el reporte dejaría de coincidir con el de otras áreas.
> **Normalizar es elegir una forma de escribir y usarla siempre.**

In [ ]:
anotar('2 · Texto', 'categoria y tipo normalizados (strip + mayúsculas)', df)

Fijate que el TPV **no se movió**: normalizar texto no cambia la suma. Pero sin este paso, **ningún `groupby` por categoría servía**.
No toda limpieza mueve el total; algunas arreglan el desglose.

---
# Paso 5 · Montos imposibles: ¿error o cliente grande?

### 5a · Ceros y negativos

In [ ]:
print(f"Montos en cero:   {(df['monto_ars'] == 0).sum()}")
print(f"Montos negativos: {(df['monto_ars'] < 0).sum()}")

df[df['monto_ars'] < 0].head()

**No hay compras de $0.** Y un negativo **podría** ser una devolución… pero estas figuran como compras aprobadas,
en categorías de consumo, y el archivo no tiene ningún campo que diga "devolución". Decisión: **las sacamos del TPV
y las mandamos a revisar** con operaciones.

Ahora, una trampa. Para sacarlas, lo natural es quedarse con los montos mayores a cero:

In [ ]:
print(f"Filas hoy:                               {len(df):,}")
print(f"Filas con df[df['monto_ars'] > 0]:       {len(df[df['monto_ars'] > 0]):,}")
print(f"Filas sin los ceros ni los negativos:    {len(df[~(df['monto_ars'] <= 0)]):,}")

**`monto_ars > 0` también borra los 2.026 nulos**, porque un nulo no es mayor que cero. Sin avisar, igual que el `inner` de la Clase 4.

La forma correcta es decir exactamente lo que querés sacar: *"todo lo que **no** sea menor o igual a cero"*.

In [ ]:
a_revisar = df[df['monto_ars'] <= 0]
df = df[~(df['monto_ars'] <= 0)].copy()

print(f'{len(a_revisar)} filas apartadas para revisar · quedan {len(df):,}')
anotar('3 · Ceros y negativos', 'Fuera del TPV y a revisar con operaciones', df)

### 5b · Los montos gigantes

El máximo es $57,5 M. ¿Es mucho? Comparémoslo con el resto de la distribución:

In [ ]:
p99 = df['monto_ars'].quantile(0.99)

print(f"Mediana: ${df['monto_ars'].median():>14,.0f}")
print(f"p99:     ${p99:>14,.0f}")
print(f"Máximo:  ${df['monto_ars'].max():>14,.0f}  ← {df['monto_ars'].max() / p99:.0f} veces el p99")

Si el máximo fuera 2 o 3 veces el p99, sería **cola larga**: pasa con plata, siempre hay alguien que gasta mucho.
**137 veces** no es cola larga.

La receta de manual para esto es el **rango intercuartil (IQR)**: todo lo que esté más allá de `Q3 + 3 × IQR` es outlier.
Probémosla:

In [ ]:
Q1, Q3 = df['monto_ars'].quantile([0.25, 0.75])
limite = Q3 + 3 * (Q3 - Q1)

fuera_iqr = df[df['monto_ars'] > limite]
print(f"Límite IQR: ${limite:,.0f}  →  {len(fuera_iqr):,} 'outliers'")
print()
print(fuera_iqr['categoria'].value_counts(normalize=True).head().round(2))

**1.089 transacciones "raras", y casi la mitad son Viajes y un cuarto Transferencias.** No son errores: es gente
comprando pasajes y PyMEs moviendo plata. El IQR marca como raro todo lo que es **grande**, y en un negocio de medios de pago
**lo grande es tu mejor cliente**.

> Borrar outliers sin preguntar es la forma más rápida de sacar del análisis justo a los clientes que más importan.

La pregunta útil no es *"¿este monto es grande?"* sino **"¿este monto es normal para *este* cliente?"**.
Para eso comparamos cada transacción con la **mediana de su propio cliente**.

`transform` es como un `groupby`, pero en vez de devolver una fila por cliente, **le pega a cada transacción el valor de su cliente**:

In [ ]:
mediana_cliente = df.groupby('cliente_id')['monto_ars'].transform('median')
df['veces_su_mediana'] = df['monto_ars'] / mediana_cliente

sospechosas = df[df['veces_su_mediana'] > 20]
print(f"{len(sospechosas)} transacciones de más de 20 veces lo que gasta habitualmente ese cliente")

(sospechosas
 .sort_values('monto_ars', ascending=False)
 [['transaccion_id', 'cliente_id', 'categoria', 'monto_ars', 'veces_su_mediana']]
 .round(0)
 .head(10))

**Ahora sí.** Una compra de ropa de $57 M, **728 veces** lo que ese cliente gasta normalmente. Un supermercado de $42 M.
Son **ceros de más**: alguien cargó el monto sin la coma decimal, o en centavos.

¿Qué hacemos? **Sabemos que hubo una compra. Lo que no sabemos es de cuánto.** Entonces el monto pasa a ser un dato faltante
(se trata en el paso que viene), y la lista va a revisión.

In [ ]:
df.loc[df['veces_su_mediana'] > 20, 'monto_ars'] = np.nan
df = df.drop(columns='veces_su_mediana')

anotar('4 · Montos gigantes', 'Más de 20× la mediana del cliente: monto desconocido (nulo)', df)

---
# Paso 6 · Nulos: la decisión depende de la pregunta

Tres columnas con huecos. Para cada una, la pregunta no es *"¿cómo lo relleno?"* sino **"¿esta fila me sirve para lo que quiero contestar?"**

In [ ]:
df[['monto_ars', 'categoria', 'fecha']].isna().sum()

| Columna | ¿Sirve para el TPV anual? | Decisión |
|---|---|---|
| `fecha` | **Sí.** Sabemos que es de 2024, aunque no de qué mes | Se queda. Para una serie **mensual** no serviría: ahí se excluye |
| `categoria` | **Sí.** El monto está | Se queda, como `Sin categoría` |
| `monto_ars` | **Es justamente lo que falta** | La decisión difícil |

Con el monto hay dos caminos. **Eliminar** las filas parece neutral, pero no lo es: al sumar, pandas **ignora los nulos**,
y eso es lo mismo que decir que esas 2.052 compras **valieron cero**.

In [ ]:
df['categoria'] = df['categoria'].fillna('Sin categoría')

print(f"TPV ignorando los montos que faltan: ${df['monto_ars'].sum() / 1e6:,.0f} M")
print(f"Compras sin monto: {df['monto_ars'].isna().sum():,} ({df['monto_ars'].isna().mean():.1%} de las filas)")

Para un **total**, dejar afuera el 7% de las compras subestima el total. El otro camino es **imputar**: estimar el monto que falta.

¿Con qué? Con el **promedio de su categoría**: una compra sin monto en Viajes probablemente se parece a las otras compras de Viajes.

> **¿Promedio o mediana?** Acá el promedio, y es la lección de la Clase 2 al revés: los montos son asimétricos, y la mediana
> describe mejor al cliente **típico**. Pero para estimar un **total**, lo que necesitás es el valor **esperado**, y ese es el promedio.
> Con la mediana, el total queda corto.

Y una regla sin excepciones: **si imputás, lo marcás.** Así cualquiera puede separar lo observado de lo estimado.

In [ ]:
df['monto_imputado'] = df['monto_ars'].isna()

promedio_categoria = df.groupby('categoria')['monto_ars'].transform('mean')
df['monto_ars'] = df['monto_ars'].fillna(promedio_categoria)

print(f"Montos imputados: {df['monto_imputado'].sum():,}")
print(f"Nulos que quedan en monto: {df['monto_ars'].isna().sum()}")
anotar('5 · Montos faltantes', 'Imputados con el promedio de su categoría (marcados)', df)

---
# Paso 7 · ⏱ Tres analistas, un mismo archivo

El mismo CSV le llegó a tres analistas el mismo lunes:

- **A** no limpió nada: sumó y mandó
- **B** aplicó la receta de manual: sacó duplicados, **borró toda fila con algún nulo**, los montos no positivos y **los outliers por IQR**
- **C** somos nosotros: limpiamos con criterio de negocio, paso por paso

Y esta vez tenemos algo que en la vida real casi nunca hay: **el número conciliado por Contabilidad**, la tabla prolija de las clases 3 y 4.

In [ ]:
referencia = pd.read_csv(DATOS + 'transacciones.csv')['monto_ars'].sum()

# Analista A: no limpia
tpv_a = sucio['monto_ars'].sum()

# Analista B: la receta de manual
b = sucio.drop_duplicates().dropna()
b = b[b['monto_ars'] > 0]
q1, q3 = b['monto_ars'].quantile([0.25, 0.75])
b = b[b['monto_ars'] <= q3 + 3 * (q3 - q1)]
tpv_b = b['monto_ars'].sum()

# Analista C: nosotros
tpv_c = df['monto_ars'].sum()

tres = pd.DataFrame({
    'TPV ($M)': [tpv_a / 1e6, tpv_b / 1e6, tpv_c / 1e6],
    'vs. Contabilidad': [f'{v / referencia - 1:+.1%}' for v in (tpv_a, tpv_b, tpv_c)],
}, index=['A · No limpia', 'B · Receta de manual', 'C · Con criterio'])
tres['TPV ($M)'] = tres['TPV ($M)'].round(0)

print(f'Contabilidad: ${referencia / 1e6:,.0f} M')
tres

In [ ]:
colores = ['firebrick', 'darkorange', 'steelblue']
plt.barh(tres.index, tres['TPV ($M)'], color=colores)
plt.axvline(referencia / 1e6, color='black', linestyle='--', linewidth=1.5, label='Contabilidad')
plt.gca().invert_yaxis()
plt.title('TPV 2024: el mismo archivo, tres analistas')
plt.xlabel('Millones de pesos')
plt.legend()
plt.show()

**La lectura que da sentido a la clase:**

- **A** informa **244 millones que no existen**. La mayor parte sale de **26 filas** con ceros de más: menos del 0,1% del archivo
- **B** se pasa para el otro lado y **pierde un tercio del negocio**. Borró 1 de cada 7 filas por tener *algún* nulo (aunque fuera la categoría), y con el IQR se llevó los viajes y las transferencias grandes
- **C** queda prácticamente clavado. Parte es mérito y parte es suerte (los datos faltantes estaban repartidos al azar), pero la diferencia con A y B no es suerte: **es criterio**

> Los tres escribieron código que corre. Los tres pueden defender su número con una tabla prolija.
> **La diferencia entre los tres no está en pandas: está en las preguntas que se hicieron antes de borrar.**

---
# Paso 8 · La bitácora y el mensaje para el directorio

Nuestro número no es "el dato crudo": **no existe tal cosa**. Es el resultado de cinco decisiones.
Por eso se documentan, para que otro pueda reconstruir el camino y decidir distinto si no está de acuerdo.

In [ ]:
pd.DataFrame(bitacora)

Cada fila es una decisión, y al lado, cuánto movió el número. Esto es lo que se adjunta cuando alguien pregunta
*"¿y este número de dónde sale?"*.

### Del análisis a la recomendación — tres bullets, no una tabla

1. **El número:** el TPV 2024 es de **$1.774 M**. El archivo del procesador, sin limpiar, dice $2.018 M: tiene **869 compras cargadas dos veces** y **26 montos con ceros de más** (hasta 700 veces lo habitual del cliente) que inflan el total en un 14%
2. **Lo que decidimos:** eliminamos los duplicados, apartamos 29 montos en cero o negativos para revisar, y **estimamos 2.052 montos que faltaban** (7%) con el promedio de su categoría. Están marcados
3. **La acción:** pedirle a operaciones que revise los 29 montos en cero o negativos y las 26 compras sospechosas, y a IT que **frene en la carga** cualquier monto que supere 20 veces el habitual del cliente. Es más barato atajarlo ahí que en cada reporte

**Escribamos los tres bullets juntos, en voz alta, con los números que acabamos de sacar.**

---
## Hasta acá la práctica guiada

**Lo que hiciste hoy:** corriste el checklist de diagnóstico, encontraste duplicados, texto inconsistente, montos imposibles y nulos,
y tomaste una decisión de negocio para cada uno. Viste que la receta de manual puede dejarte peor que no limpiar,
y dejaste cada decisión anotada en una bitácora.

**Lo que sigue:** abrí la notebook **`Clase_5_Practica_Individual.ipynb`**.
Mismo archivo, preguntas nuevas, y esta vez lo resolvés vos con Gemini de copiloto.

> Si querés quedarte con tu trabajo: **`Archivo → Guardar una copia en Drive`**.